# PPE Detection — YOLOv8s Optimize Eğitim

## 1) GPU & Dependencies Kurulumu

In [ ]:
!pip install ultralytics pyyaml -q

import torch
import os

print("=" * 70)
print("GPU KONTROL")
print("=" * 70)

gpu_count = torch.cuda.device_count()
print(f"GPU Sayısı: {gpu_count}")

if gpu_count == 0:
    print("\n -> GPU BULUNAMADI")
    print("Notebook ayarlarından Accelerator → GPU (T4 x2) seçilmeli.")
    exit(1)

for i in range(gpu_count):
    gpu_name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"->GPU {i}: {gpu_name} ({mem:.1f} GB)")

print("\n-> GPU HAZIR")
print("=" * 70)

## 2) Dosya Unzip & Hedef "best.pt" Bulma

In [ ]:
import os
import shutil
from pathlib import Path

print("=" * 70)
print("-> DOSYA UNZIPPING")
print("=" * 70)

input_dir = Path("/kaggle/input")
working_dir = Path("/kaggle/working")

dataset_found = False
best_pt_found = False

for root, dirs, files in os.walk(input_dir):
    # bize ait merged_dataset bulunur
    if "merged_dataset" in dirs:
        src_dataset = os.path.join(root, "merged_dataset")
        print(f"-> 'merged_dataset' bulundu: {src_dataset}")
        shutil.copytree(src_dataset, working_dir / "merged_dataset", dirs_exist_ok=True)
        dataset_found = True
        
    # merged_dataset'e ait "best.pt" aranır
    if "best.pt" in files:
        # sadece "yolov8m_ppe_v2" veya "notebookddc" geçen, bize ait model seçilir
        if "yolov8m_ppe_v2" in root or "notebookddc" in root or "_output_" in root:
            src_weights = os.path.join(root, "best.pt")
            print(f"-> İstenen best.pt bulundu: {src_weights}")
            
            target_weights_dir = working_dir / "results/yolov8m_ppe_v2/weights"
            os.makedirs(target_weights_dir, exist_ok=True)
            shutil.copy2(src_weights, target_weights_dir / "best.pt")
            best_pt_found = True
        else:
            # sonradan eklenen dataset ağırlıkları es geçilir
            print(f"-> Yabancı best.pt es geçildi: {os.path.join(root, 'best.pt')}")

    print("\n-> Tüm veriler tam isabetle çalışma alanına alındı.")

## 3) Eğitim Tahminleri

In [ ]:
import shutil
from pathlib import Path

print("=" * 70)
print("-> TRAIN SETİ ÜZERİNDE TAHMİNLER ALINIYOR...")
print("=" * 70)

# zipten çıkan best.pt yolu
best_model_path = '/kaggle/working/results/yolov8m_ppe_v2/weights/best.pt'

!yolo task=detect mode=predict \
  model={best_model_path} \
  source=/kaggle/working/merged_dataset/train/images \
  conf=0.3 \
  save=True \
  save_txt=True \
  project=/kaggle/working/results \
  name=mining_predictions

print("-> Tahminler Kaydedildi. Karşılaştırma yapılıyor.")

## 4) False Negative Mining

In [ ]:
import os
import shutil
from pathlib import Path

gt_labels_dir = Path("/kaggle/working/merged_dataset/train/labels")
gt_images_dir = Path("/kaggle/working/merged_dataset/train/images")
pred_labels_dir = Path("/kaggle/working/results/mining_predictions/labels") 

OVERSAMPLE_COUNT = 2 
fn_sayisi = 0

print("=" * 70)
print("-> FALSE NEGATIVE MINING")
print("=" * 70)

for gt_label_file in gt_labels_dir.glob("*.txt"):
    filename = gt_label_file.name
    image_filename = gt_label_file.stem + ".jpeg"
    
    gt_image_file = gt_images_dir / image_filename
    pred_label_file = pred_labels_dir / filename
    
    is_false_negative = False
    
    if not pred_label_file.exists():
        is_false_negative = True
    else:
        with open(gt_label_file, "r") as f:
            gt_lines = f.readlines()
        with open(pred_label_file, "r") as f:
            pred_lines = f.readlines()
            
        if len(gt_lines) > len(pred_lines):
            is_false_negative = True

    if is_false_negative:
        fn_sayisi += 1
        for i in range(OVERSAMPLE_COUNT):
            new_img_name = f"{gt_image_file.stem}_fn_copy{i}{gt_image_file.suffix}"
            new_lbl_name = f"{gt_label_file.stem}_fn_copy{i}{gt_label_file.suffix}"
            
            if gt_image_file.exists():
                shutil.copy(str(gt_image_file), str(gt_images_dir / new_img_name))
                shutil.copy(str(gt_label_file), str(gt_labels_dir / new_lbl_name))

print(f"-> İşlem tamamlandı. Toplam {fn_sayisi} adet zorlu resim (False Negative) tespit edildi.")
print(f"-> Bu resimler {OVERSAMPLE_COUNT} kez kopyalanarak 'merged_dataset' güncellendi.")

## 5) Fine Tuning

In [ ]:
import os

print("=" * 70)
print("==== YOLO v8m FINE-TUNING (En Güncel Dataset) ====")
print("=" * 70)

os.chdir("/kaggle/working")
# Zipten çıkan eski modelimiz
best_model_path = '/kaggle/working/results/yolov8m_ppe_v2/weights/best.pt'

!yolo task=detect mode=train \
  model={best_model_path} \
  data=/kaggle/working/merged_dataset/data.yaml \
  epochs=30 \
  imgsz=640 \
  batch=16 \
  device=0,1 \
  lr0=0.001 \
  weight_decay=0.0005 \
  project=/kaggle/working/results \
  name=yolov8m_ppe_finetuned \
  save=True

print("\n-> NİHAİ EĞİTİM TAMAMLANDI.")